<small>DAY 51 &nbsp; / &nbsp; 01</small>

# Docker & Docker Compose

Containers, images, networks, and persistent data.

**A small application, from one container to two services.**

Read one slide at a time. Run terminal commands only when you reach the practical steps.


<small>DAY 51 &nbsp; / &nbsp; 02</small>

# Why use containers?

An application needs more than source code: a runtime, libraries, and configuration.

- Package the app and its dependencies together.
- Start separate environments without installing every tool directly on your machine.
- Share the same image across development and deployment.

The image standardizes the application environment. Host architecture, configuration, and external systems still matter.


<small>DAY 51 &nbsp; / &nbsp; 03</small>

# What is a container?

A container runs an isolated process, or a small group of related processes.

It has its own view of files, processes, and networking. It shares the Docker host's operating-system kernel.

**Docker** provides tools to build images and create, run, and manage containers.

[Container basics](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/)


<small>DAY 51 &nbsp; / &nbsp; 04</small>

# Containers and virtual machines

A VM includes a guest operating system. A container packages the application environment.

On Windows and macOS, Docker Desktop normally runs Linux containers inside a Linux VM.

![Comparison of virtual-machine and container layers](assets/containers-vs-vms.svg)


<small>DAY 51 &nbsp; / &nbsp; 05</small>

# A quick comparison

| | Virtual machine | Container |
|---|---|---|
| Isolation boundary | Virtual hardware and guest OS | Processes sharing a kernel |
| Startup | Includes guest OS boot | Starts application processes |
| Typical overhead | Higher | Lower |
| OS choice | Separate guest OS per VM | Requires a compatible kernel |
| Useful for | Full machines, different OS environments | Apps, databases, repeatable jobs |

Containers can run inside VMs. Both are useful.


<small>DAY 51 &nbsp; / &nbsp; 06</small>

# Image to container

An **image** is the reusable starting filesystem and configuration.
A **container** is an instance created from it, with its own writable layer.

![Dockerfile builds an image; the image starts two independent containers](assets/image-to-container.svg)


<small>DAY 51 &nbsp; / &nbsp; 07</small>

# Four names to recognize

| Term | Meaning | Example |
|---|---|---|
| Dockerfile | Instructions for building an image | `FROM`, `COPY`, `CMD` |
| Image | Packaged application environment | `postgres:16` |
| Registry | Server that stores and distributes images | Docker Hub |
| Container | An instance created from an image | A running PostgreSQL server |

In `postgres:16`, `postgres` is the image repository and `16` is a tag.
Tags can change; a digest identifies exact image content.

[Docker overview](https://docs.docker.com/get-started/docker-overview/)


<small>DAY 51 &nbsp; / &nbsp; 08</small>

# Install Docker on the remote VM

**Remote Ubuntu VM only. Do not run these installation steps in WSL or on a corporate laptop.**

Connect to the remote VM and run these commands in its terminal:

```bash
sudo apt update
sudo apt install -y docker.io docker-compose-v2
```

These Ubuntu packages install Docker Engine and the Docker Compose plugin.
Use `docker compose` with a space.


<small>DAY 51 &nbsp; / &nbsp; 09</small>

# Start Docker and enable access

On the same remote VM:

```bash
sudo systemctl enable --now docker
sudo usermod -aG docker "$USER"
newgrp docker
```

- `enable --now` starts Docker and enables it after a reboot.
- `usermod` adds your login user to the Docker group.
- `newgrp` opens a shell with the new group membership. Continue in that shell, or log out and reconnect.

Docker group membership gives root-level control of this VM.


<small>DAY 51 &nbsp; / &nbsp; 10</small>

# Verify the installation

Run these in the remote VM terminal:

```bash
docker --version
docker compose version
docker run --rm hello-world
```

The first two commands show installed versions. The last command downloads a test image, prints a success message, and removes the test container after it exits.

If Docker reports permission denied, reconnect to the VM and retry. If it cannot connect to the daemon, check `sudo systemctl status docker`.


<small>DAY 51 &nbsp; / &nbsp; 11</small>

# Open the remote web pages locally

The examples publish web ports on the VM's loopback address.
Use SSH forwarding to open them in your local browser.

From your local terminal, replace the user and VM address:

```bash
ssh -L 8080:127.0.0.1:8080 -L 8081:127.0.0.1:8081 user@vm-address
```

Keep this connection open. Run Docker commands on the remote VM; open `http://localhost:8080` or `http://localhost:8081` in your local browser when instructed.

No local Docker installation is required.


<small>DAY 51 &nbsp; / &nbsp; 12</small>

# Run a small web server

```sh
docker run -d --name hello-web -p 127.0.0.1:8080:80 nginx:alpine
```

Open **http://localhost:8080**. You should see the Nginx welcome page.

- `-d`: run in the background.
- `--name`: give the container a readable name.
- `-p`: forward a host port to a container port.
- `nginx:alpine`: use this image; Docker pulls it if needed.


<small>DAY 51 &nbsp; / &nbsp; 13</small>

# Read the port mapping

```text
127.0.0.1:8080:80
          |    +-- Port inside the container
          +------- Port on the Docker host (VM)
127.0.0.1 -------- Listen on local loopback only
```

Your browser uses **localhost:8080**; Nginx listens on **80** inside the container.

A published port provides a route from the host. `EXPOSE` in a Dockerfile alone does not publish one.

[Port publishing](https://docs.docker.com/engine/network/port-publishing/)


<small>DAY 51 &nbsp; / &nbsp; 14</small>

# Start, stop, and remove

`docker run` creates a new container. `docker start` restarts an existing stopped container.

The container stops when its main process exits.

![Container lifecycle: image, running container, stopped container](assets/container-lifecycle.svg)


<small>DAY 51 &nbsp; / &nbsp; 15</small>

# A few commands are enough

| Command | Purpose |
|---|---|
| `docker ps` | List running containers |
| `docker ps -a` | Include stopped containers |
| `docker logs hello-web` | Read application output |
| `docker exec -it hello-web sh` | Open a shell; type `exit` to leave |
| `docker stop hello-web` | Stop it |
| `docker rm hello-web` | Remove it after stopping |

If the name already exists, restart that container or remove it before running another with the same name.


<small>DAY 51 &nbsp; / &nbsp; 16</small>

# Build your own image

A minimal Dockerfile for a folder containing `index.html`:

```dockerfile
FROM nginx:alpine
COPY index.html /usr/share/nginx/html/index.html
```

Build from that folder:

```sh
docker build -t my-site:1 .
```

`.` is the **build context**. Use `.dockerignore` to exclude unnecessary files and secrets. This image inherits Nginx's startup command.

[Writing a Dockerfile](https://docs.docker.com/get-started/docker-concepts/building-images/writing-a-dockerfile/)


<small>DAY 51 &nbsp; / &nbsp; 17</small>

# Where should data live?

| Storage | Useful for | When the container is removed |
|---|---|---|
| Container writable layer | Temporary application files | Removed with the container |
| Named volume | Database files and persistent data | Remains until explicitly removed |
| Bind mount | A specific host folder, such as source code | Host files remain |

Stopping a container does not remove its writable layer. Removing it does.

A volume is persistent storage, **not a backup**.

[Docker volumes](https://docs.docker.com/engine/storage/volumes/)


<small>DAY 51 &nbsp; / &nbsp; 18</small>

# Keep data outside the container

Mount a named volume at the path where the application writes its data.
A replacement container can mount that same volume.

![Old and new PostgreSQL containers use the same persistent volume](assets/persistent-volume.svg)


<small>DAY 51 &nbsp; / &nbsp; 19</small>

# How containers find each other

Containers on the same user-defined network can use container names or network aliases.

Compose creates a project network by default and makes services discoverable by their service names.

**Inside a container, `localhost` means that container itself.**
Use `db`, not `localhost`, when another container needs the database.

[Compose networking](https://docs.docker.com/compose/how-tos/networking/)


<small>DAY 51 &nbsp; / &nbsp; 20</small>

# What Docker Compose adds

Compose describes a multi-container application in a **`compose.yaml`** file.

- **Services** define application components and how to run them.
- **Networks** connect components.
- **Volumes** keep persistent data.

A service is a definition; containers are the running instances of that definition.
One command can create the application's containers and supporting resources.

[What is Docker Compose?](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-docker-compose/)


<small>DAY 51 &nbsp; / &nbsp; 21</small>

# Read YAML by its indentation

```yaml
services:                    # A mapping
  adminer:                   # A service name
    image: adminer:4          # A key and value
    ports:                   # A list
      - "127.0.0.1:8081:8080" # One list item
```

Use **spaces, not tabs**. Indentation defines which settings belong together.
`key: value` defines an entry; `-` starts a list item; `#` begins a comment.

Quote port mappings. Current Compose files do not need a top-level `version:` field.

[Compose file format](https://docs.docker.com/reference/compose-file/version-and-name/)


<small>DAY 51 &nbsp; / &nbsp; 22</small>

# Two services, one application

**Adminer** provides a browser interface. **PostgreSQL** stores the data.

Only Adminer publishes a host port. The database is reached over the internal project network.

![Browser connects to Adminer, which connects to PostgreSQL by service name](assets/network-and-ports.svg)


<small>DAY 51 &nbsp; / &nbsp; 23</small>

# Compose: the database service

The complete file is supplied in **`compose-demo/compose.yaml`**. This is its database section:

```yaml
services:
  db:
    image: postgres:16
    environment:
      POSTGRES_USER: learner
      POSTGRES_PASSWORD: local-demo-only
      POSTGRES_DB: practice
    volumes:
      - pgdata:/var/lib/postgresql/data
```

`pgdata` is the named volume; the path is inside the PostgreSQL 16 container.
The password is for this local demo. Do not reuse it for a deployed database.


<small>DAY 51 &nbsp; / &nbsp; 24</small>

# Compose: the browser service

The same file continues with another service and declares the storage resource:

```yaml
  adminer:
    image: adminer:4
    ports:
      - "127.0.0.1:8081:8080"
    depends_on:
      db:
        condition: service_healthy

volumes:
  pgdata:
```

`adminer` sits under `services`. The final `volumes` key sits at the top level.
The file uses Compose's default network; an explicit `networks` section is optional.


<small>DAY 51 &nbsp; / &nbsp; 25</small>

# Started does not mean ready

The supplied database service includes this readiness check:

```yaml
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U learner -d practice"]
      interval: 5s
      timeout: 3s
      retries: 10
```

`condition: service_healthy` waits for the database check to pass before starting Adminer.
Plain `depends_on` controls startup order, not readiness. Applications still need to handle later connection failures.

[Startup order and health checks](https://docs.docker.com/compose/how-tos/startup-order/)


<small>DAY 51 &nbsp; / &nbsp; 26</small>

# Start the Compose example

Copy the `Day51` folder to the remote VM. In its terminal, open **Day51**, then run:

```sh
cd compose-demo
docker compose config -q
docker compose up -d
docker compose ps
```

The first start downloads images. Wait until `db` is healthy and `adminer` is running.

Open **http://localhost:8081**. Choose **PostgreSQL**, then enter server **db**, username **learner**, password **local-demo-only**, and database **practice**.


<small>DAY 51 &nbsp; / &nbsp; 27</small>

# Check that the data survives

In Adminer's **SQL command** screen, run:

```sql
CREATE TABLE IF NOT EXISTS notes (message text);
INSERT INTO notes VALUES ('Stored in a Docker volume');
SELECT * FROM notes;
```

From the same `compose-demo` folder:

```sh
docker compose down
docker compose up -d
```

Log in again and run `SELECT * FROM notes;`. The row remains because `pgdata` was reused.


<small>DAY 51 &nbsp; / &nbsp; 28</small>

# Inspect and clean up

Run these from the `compose-demo` folder:

| Command | Result |
|---|---|
| `docker compose logs db` | Read database startup messages |
| `docker compose stop` | Stop services; retain containers and data |
| `docker compose start` | Start those stopped containers |
| `docker compose down` | Remove project containers and network; retain the named volume |
| `docker compose down -v` | Also delete this project's named volume and its database data |

Use `down` for normal cleanup. Use `down -v` only when you want to erase this demo's data.


<small>DAY 51 &nbsp; / &nbsp; 29</small>

# Common problems, quick checks

| Symptom | Check |
|---|---|
| Cannot connect to Docker | On the VM, check `sudo systemctl status docker` |
| Port already in use | Change host port: `127.0.0.1:8082:8080` |
| Container exits immediately | Read `docker logs` or `docker compose logs` |
| Database connection fails | Use server `db`; check the database is healthy |
| New password has no effect | PostgreSQL initialization variables apply to an empty data directory |
| YAML does not load | Check indentation; run `docker compose config -q` |

For a running database, change credentials in PostgreSQL. Recreating a container does not reinitialize its existing volume.


<small>DAY 51 &nbsp; / &nbsp; 30</small>

# Habits to keep

- Keep one application responsibility per service.
- Save Compose files and Dockerfiles with the project.
- Store persistent data in a volume and back it up separately.
- Keep secrets out of images and source control; use a suitable secrets mechanism for deployments.
- Publish only the ports you need; use loopback for local examples.
- Set resource limits when workloads compete for CPU or memory.

Compose is useful for local development and single-host applications. Multi-host scheduling requires an orchestrator.


<small>DAY 51 &nbsp; / &nbsp; 31</small>

# Check your understanding

1. Can two containers be created from one image?
2. Which address does Adminer use to reach PostgreSQL?
3. What survives `docker compose down` in this example?
4. Why does the database need a health check?

<details><summary>Answers</summary>

1. Yes; each has its own process environment and writable layer.
2. `db:5432`, over the project network.
3. The named volume and database files. Local images also remain.
4. Starting the database process does not mean it is ready to accept connections.

</details>
